# Publication study — GPU runner (Google Colab)

Runs `research/` training and evaluation from the `for-publishing` branch. Runs are written to
Google Drive, so a disconnect loses at most the current evaluation interval: re-running the
launch cell skips finished runs (`done.json`).

Runtime → Change runtime type → **GPU** (T4 is enough). Then run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/wild-drone-research/runs'   # change if you like
!mkdir -p {RUNS}

In [ ]:
!rm -rf /content/wild-drone-appeared
!git clone --depth 1 -b for-publishing https://github.com/SayuruRehan/wild-drone-appeared /content/wild-drone-appeared
%cd /content/wild-drone-appeared
!git log --oneline -1
!pip -q install -e . pytest

In [ ]:
import torch, os
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
os.environ['PYTHONPATH'] = '/content/wild-drone-appeared/research:/content/wild-drone-appeared/src'
!python -m pytest -q research/tests

## Choose an experiment

Edit `EXPERIMENT` below. `PARALLEL` runs share the GPU; 3 is a good start on a T4 (the
environment itself runs on the CPU, and Colab has only 2 vCPUs, so more rarely helps).

In [ ]:
EXPERIMENT = 'pilot-main40'     # folder name under RUNS
ENV = 'main40'                    # 'dev20' or 'main40'
KINDS = 'joint vdn dcg structured structured+'
SEEDS = '0'
STEPS = 300000
TRAIN_DRONES = '3'
PARALLEL = 3
EXTRA = '--eval-every 25000 --eval-worlds 10 --eval-drones 3 5 --device auto'

In [ ]:
!python research/scripts/pilot.py --kinds {KINDS} --seeds {SEEDS} --steps {STEPS} \
    --env {ENV} --train-drones {TRAIN_DRONES} --parallel {PARALLEL} \
    --root {RUNS}/{EXPERIMENT} -- {EXTRA}

## Watch progress

Colab runs cells one at a time, so this cell waits until training finishes. **While training is running, open Terminal (bottom bar) instead** and run:

```bash
tail -n 1 /content/drive/MyDrive/wild-drone-research/runs/<EXPERIMENT>/*/log.jsonl
tail -n 5 /content/drive/MyDrive/wild-drone-research/runs/<EXPERIMENT>/*/stdout.txt
```

Each run logs a line every `--eval-every` steps; `minutes` at 25k × 12 ≈ time for 300k steps.

In [ ]:
import json, glob
for log in sorted(glob.glob(f'{RUNS}/{EXPERIMENT}/*/log.jsonl')):
    lines = [json.loads(l) for l in open(log)]
    last = lines[-1]
    print(log.split('/')[-2].ljust(16), {k: round(v, 4) if isinstance(v, float) else v
          for k, v in last.items() if k.startswith(('steps', 'dev_fresh', 'minutes'))})

## Evaluate on development worlds across fleet sizes

Uses the baseline episodes committed in `research/results/headroom` for the same worlds.

In [ ]:
!python research/scripts/evaluate_runs.py --root {RUNS}/{EXPERIMENT} --drones 3 4 5 6 8 \
    --worlds 20 --workers 2 --out {RUNS}/{EXPERIMENT}-eval
!cat {RUNS}/{EXPERIMENT}-eval/table.md

Copy `table.md` and `episodes.csv` from the `-eval` folder back into
`research/results/<experiment>/` in the repository (or send them to Claude) so they are
versioned with the code that produced them. Do **not** pass `--final` until the protocol
is frozen.